# EV Charging Analysis - Organized Colab Workflow

This notebook is an end-to-end interactive workspace for:

- Running the pipeline in order
- Inspecting intermediate tables quickly
- Visualizing feature importance and spatial diagnostics
- Iterating on experiments without editing core pipeline scripts

Recommended runtime: **Python 3** (GPU not required).

## 1) Setup and Repo Sync

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/ev-charging-analysis.git"
REPO_DIR = Path("/content/ev-charging-analysis")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already present:", REPO_DIR)

%cd /content/ev-charging-analysis
!pip install -r requirements.txt

## 2) Optional: Mount Google Drive and Sync Data

Use this if your `data/` folder is stored in Google Drive.

In [ ]:
USE_DRIVE = False  # set True if data is in Drive
DRIVE_DATA_DIR = "/content/drive/MyDrive/ev-charging-analysis/data"

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !rsync -av --delete "{DRIVE_DATA_DIR}/" "/content/ev-charging-analysis/data/"
    print("Synced data from Drive")
else:
    print("Skipping Drive sync")

## 3) Run Pipeline in Order (One Command)

In [ ]:
# Full run: 04b -> 05 -> 06 -> 07 -> 08
!python run_pipeline.py

In [ ]:
# Quick rerun of only model + siting
# !python run_pipeline.py --from-stage 06 --to-stage 08

## 4) Load Diagnostic Helpers

In [ ]:
from notebooks.colab_interactive_diagnostics import (
    load_tables,
    show_basic_checks,
    load_metrics,
    compare_scope_metrics,
    plot_spatial_seed_pr_auc,
    plot_xgboost_gain_importance,
)

BASE_DIR = "/content/ev-charging-analysis"

## 5) Intermediate Table Inspection

In [ ]:
tables = load_tables(BASE_DIR)
show_basic_checks(tables)

# convenient handles
zcta = tables["zcta_modeling_features"]
preds = tables["predictions_zcta"]
candidates = tables["candidate_sites"]
ranked = tables["recommended_sites_topN"]

In [ ]:
# Quick quality checks
top100 = ranked[ranked["scenario_top_n"].astype(int) == 100].copy()
print("Top100 rows:", len(top100))
print("Top100 unique coordinates:", top100[["lat", "lon"]].drop_duplicates().shape[0])
print("Top100 unique candidate IDs:", top100["candidate_id"].nunique())
print("Top100 region counts:\n", top100.merge(candidates[["candidate_id", "region"]], on="candidate_id", how="left")["region"].value_counts())

## 6) Model Metrics and Spatial Generalization

In [ ]:
artifacts = load_metrics(BASE_DIR)
metrics = artifacts["metrics"]
holdout = artifacts["holdout"]

compare_scope_metrics(BASE_DIR)
print("\nHeld-out test states:", holdout["test_states"])

In [ ]:
plot_spatial_seed_pr_auc(BASE_DIR)

## 7) Feature Importance (XGBoost Gain)

In [ ]:
imp = plot_xgboost_gain_importance(BASE_DIR, top_n=20)
imp

## 8) Experiment Loop: Weight Sensitivity for Top-N

This block lets you test different scoring weights quickly without changing source files.
It imports the stage-08 module and evaluates region distribution in Top-100.

In [ ]:
import importlib.util
from pathlib import Path
import pandas as pd

spec = importlib.util.spec_from_file_location("stage08", "/content/ev-charging-analysis/notebooks/08_site_ranking_topN.py")
stage08 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(stage08)

processed = Path("/content/ev-charging-analysis/data/processed")
cand = pd.read_parquet(processed / "candidate_sites.parquet")
zall = pd.read_parquet(processed / "zcta_modeling_features.parquet")
cols = [
    "ZIP_ZCTA", "final_lat", "final_lon", "total_population",
    "median_household_income", "nearest_dcfc_miles",
    "weather_extreme_heat_days_year", "weather_heavy_precip_days_year",
]
z = zall[cols].copy()
if "is_continental_us" in zall.columns:
    z = zall[zall["is_continental_us"] == 1][cols].copy()
z = z[z["final_lat"].notna() & z["final_lon"].notna()]

weight_grid = [
    (0.30, 0.42, 0.08, 0.20),
    (0.35, 0.38, 0.10, 0.17),
    (0.25, 0.50, 0.05, 0.20),
]

rows = []
for w_gap, w_need, w_pop, w_eq in weight_grid:
    stage08.W_GAP, stage08.W_NEED, stage08.W_POP, stage08.W_EQUITY = w_gap, w_need, w_pop, w_eq
    scored = stage08.score_candidates(cand, z).sort_values("composite_score", ascending=False).head(100)
    merged = scored.merge(cand[["candidate_id", "region"]], on="candidate_id", how="left")
    counts = merged["region"].value_counts().to_dict()
    rows.append({
        "W_GAP": w_gap,
        "W_NEED": w_need,
        "W_POP": w_pop,
        "W_EQUITY": w_eq,
        "West": counts.get("West", 0),
        "Midwest": counts.get("Midwest", 0),
        "South": counts.get("South", 0),
        "Northeast": counts.get("Northeast", 0),
    })

pd.DataFrame(rows).sort_values(["West", "Midwest"], ascending=False)

## 9) Save Key Tables for Reporting

In [ ]:
out_dir = Path("/content/ev-charging-analysis/reports/diagnostics")
out_dir.mkdir(parents=True, exist_ok=True)

top100.to_csv(out_dir / "top100_sites_snapshot.csv", index=False)
pd.Series(metrics).to_json(out_dir / "model_metrics_snapshot.json", indent=2)
print("Saved diagnostics to:", out_dir)

## 10) Next Steps

- If you change core logic in `06` or `08`, rerun `run_pipeline.py --from-stage 06 --to-stage 08`
- Reopen Streamlit locally to inspect maps and threshold diagnostics
- Keep this notebook as the experiment log for reproducible comparisons